# Ensemble agent (notebook prototype)

This notebook walks through **data → Chroma retrieval → RAG-style prompting → frontier LLM → Modal specialist pricer** and benchmarks on the Hugging Face test split.

**Working directory:** run Jupyter with the **project root** or **`notebooks/`** as the current working directory. The next cell sets `root_dir = Path.cwd().resolve().parents[0]` when CWD is `notebooks/` (parent = repo root). If you launch from repo root, change that line to `root_dir = Path.cwd().resolve()` so `src/` and paths still resolve.

**Env / secrets**
- `HF_TOKEN` — Hugging Face (dataset + hub)
- `OPENAI_API_KEY` — only for the **Frontier (GPT-5.1)** cells
- Modal CLI auth — for **Specialist** (`pricer`, class `Pricer`)

**Time / cost**
- Building the Chroma index over the **full training set** is slow (many encoder passes); existing non-empty collection skips rebuild.
- `evaluate(..., size=250)` on the specialist = **250 Modal remote calls**.
- Benchmarking GPT on 250 rows incurs **API cost**; use the optional cell with `RUN_GPT_BENCHMARK = False` by default.

**Suggested run order:** Setup → Load dataset → Chroma + encoder → Build collection → Similarity + RAG helpers → Frontier GPT (optional) → Evaluation imports → Specialist → Benchmarks.


## Setup


In [ ]:
from pathlib import Path
import sys

root_dir = Path.cwd().resolve().parents[0]
sys.path.insert(0, str(root_dir / "src"))
sys.path.insert(0, str(root_dir / "notebooks"))


In [ ]:
import os

from dotenv import load_dotenv
from huggingface_hub import login
from tqdm.auto import tqdm

from deal_hunter.agents.items import Item


## Load dataset


In [ ]:
load_dotenv(override=True)
hf_token = os.environ["HF_TOKEN"]
login(token=hf_token, add_to_git_credential=False)


In [ ]:
pricer_data = "Vishy08/pricer-data"
train, test_items, val = Item.from_hub(dataset_name=pricer_data)
print(
    f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test_items):,} test items"
)


## ChromaDB


In [ ]:
import chromadb

DB = str(root_dir / "notebooks" / "products_vectorstore")
chroma_client = chromadb.PersistentClient(path=DB)


### Encoder (`sentence-transformers/all-MiniLM-L6-v2`)


In [ ]:
from sentence_transformers import SentenceTransformer

encoder_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
sentences = [item.test_prompt for item in test_items[:50]]
embeddings = encoder_model.encode(sentences=sentences)


### Optional: embedding check


In [ ]:
type(embeddings)


### Build Chroma collection


In [ ]:
import re

def summarizer(item):
    question = "How much does this cost to the nearest dollar?\n\n"
    prefix = "\n\nPrice is $"
    summary = item.test_prompt.replace(question, "").replace(prefix, "")
    return f"{summary}"


def parse_price(text: object) -> float:
    s = str(text).replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0


### Optional: summarizer output sample


In [ ]:
summarizer(test_items[3000])


In [ ]:
def build_product_collection(
    chroma_client, encoder_model, items, name="products", batch_size=1000
):
    collection = chroma_client.get_or_create_collection(name)
    try:
        if collection.count() > 0:
            return collection
    except Exception:
        existing = collection.get(limit=1, include=[])
        if existing.get("ids"):
            return collection

    with tqdm(
        total=len(items), desc=f"Chroma {name}:", unit="doc", dynamic_ncols=True
    ) as pbar:
        for start in range(0, len(items), batch_size):
            batch = items[start : start + batch_size]
            docs = [summarizer(item) for item in batch]
            embeddings = encoder_model.encode(docs).astype(float).tolist()
            metas = [{"category": item.category, "price": item.price} for item in batch]
            doc_ids = [f"{name}_{start + k}" for k in range(len(batch))]
            collection.upsert(
                documents=docs,
                metadatas=metas,
                ids=doc_ids,
                embeddings=embeddings,
            )
            pbar.update(len(batch))
    return collection


In [ ]:
collection = build_product_collection(chroma_client, encoder_model, train[:])


## Similarity search


In [ ]:
chroma_client.get_collection("products")


In [ ]:
# Long product blurb (realistic length) for a manual retrieval demo
text = [
    "POWERTEC 70107V 4 T-Fitting for Dust Collection Hose, 1 PK\nIntroducing the 4-Inch T-Fitting by POWERTEC. This handy t-shaped connector optimizes the performance of your dust collection system by allowing you to add branches (connections) to your setup without compromising its efficiency. This creates a secure, airtight connection to ensure that harmful dust, dirt and debris is sucked in and expelled directly into your dust collector – resulting in clean air and a clean work environment. This dust collector hose adapter is ideal for professionals, hobbyists and workshops where space constraints or machine design are a concern. T-Shaped Fitting Dimensions 4-Inch OD nominal ends measure O.D. and I.D. Premium Build The POWERTEC T-Fitting is made out of high quality ABS plastic. This lightweight material is commonly known for it’s super durability"
]
result = collection.query(query_texts=text, n_results=3)
print(result["documents"][0][:])


In [ ]:
def find_similar(text):
    vector = encoder_model.encode(text)
    result = collection.query(
        query_embeddings=vector,
        include=["documents", "metadatas"],
        n_results=5,
    )
    document = result["documents"][0][:]
    price = [m["price"] for m in result["metadatas"][0][:]]
    return document, price


In [ ]:
find_similar(text)


## RAG-style context for LLM


In [ ]:
def make_context(similars, prices):
    message = "Here are the similar products with prices, which can be helpful for context\n"
    for similar, price in zip(similars, prices):
        message += f"Related Product:\n{similar}\nPrice is ${price}\n\n"
    return message


In [ ]:
similars, prices = find_similar(text)
context = make_context(similars, prices)
print(context)


In [ ]:
def message_for(item, similars, prices):
    message = f"Estimate the price of the product. Respond with Price only No Explanation\n\n{summarizer(item)}"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]


In [ ]:
similars, prices = find_similar(summarizer(train[2345]))
print(message_for(train[2345], similars, prices))


## GPT-5.1 + RAG


In [ ]:
from litellm import completion

# Requires OPENAI_API_KEY and earlier load_dotenv() from dataset section
_ = os.environ["OPENAI_API_KEY"]


In [ ]:
def gpt_5_1_rag(item):
    documents, prices = find_similar(summarizer(item))
    response = completion(
        model="gpt-5.1",
        messages=message_for(item, documents, prices),
        reasoning_effort="none",
        seed=42,
    )
    return response.choices[0].message.content


In [ ]:
response = gpt_5_1_rag(train[56678])
print(response)


## Evaluation utilities


In [ ]:
import importlib

import pandas as pd

import deal_hunter.services.testing as testing

importlib.reload(testing)
# Do not import ``test`` from this module — it shadows the HF ``test`` split name.
from deal_hunter.services.testing import Tester, evaluate


### Optional: benchmark GPT on the test split (API cost)


In [ ]:
RUN_GPT_BENCHMARK = False
N_EVAL_GPT = 250

if RUN_GPT_BENCHMARK:

    def gpt_5_1_rag_row(row):
        item = Item.model_validate(row.to_dict())
        return parse_price(gpt_5_1_rag(item))

    _test_df = pd.DataFrame([x.model_dump() for x in test_items]).reset_index(drop=True)
    _n = min(N_EVAL_GPT, len(_test_df))
    evaluate(gpt_5_1_rag_row, _test_df, title="GPT-5.1 RAG", size=_n)


## Specialist (Modal fine-tuned pricer)

Deployed from [`../src/deal_hunter/services/pricer.py`](../src/deal_hunter/services/pricer.py): Modal app name `pricer`, class `Pricer`.


In [ ]:
import modal

Pricer = modal.Cls.from_name("pricer", "Pricer")
pricer = Pricer()


In [ ]:
def specialist(item):
    return pricer.price.remote(summarizer(item))


In [ ]:
specialist(train[56])


## Benchmarks (test split)


In [ ]:
N_EVAL = 250
BENCHMARK_TITLE = "Llama_3.1_8B_Finetuned"


def llama8(row) -> float:
    item = Item.model_validate(row.to_dict())
    raw = specialist(item)
    return parse_price(raw)


test_df = pd.DataFrame([x.model_dump() for x in test_items]).reset_index(drop=True)
n = min(N_EVAL, len(test_df))

evaluate(llama8, test_df, title=BENCHMARK_TITLE, size=n)
